# Notebook 6 . Bagi Dataset

Pisahkan satu CSV menjadi dua atau tiga bagian acak, misalnya data latih dan uji. Rasio dapat diatur dan hasil dapat diulang dengan angka acak yang sama. Gunakan `Kolom_Grup` bila semua baris dari satu orang/dokumen harus tetap berada di bagian yang sama.

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd

# PENGATURAN YANG BISA DIUBAH
Input_CSV = Path("Input/input.csv")
Separator_CSV = ","  # Ganti menjadi ";" jika CSV memakai titik koma
Rasio_Latih = 0.8
Rasio_Validasi = 0.0  # Contoh: 0.1; bagian uji mengambil sisa proporsi
Kolom_Grup = None     # Contoh: "document_id" untuk membagi per grup, bukan per baris
Benih_Acak = 42
Folder_Output = Path("Output")

def find_project_dir():
    current = Path.cwd().resolve()
    for folder in (current, *current.parents):
        if (folder / "Input").is_dir():
            (folder / "Output").mkdir(parents=True, exist_ok=True)
            return folder
    return current

BASE_DIR = find_project_dir()


## Buat dan simpan bagian dataset

In [ ]:
if Rasio_Latih <= 0 or Rasio_Validasi < 0 or Rasio_Latih + Rasio_Validasi >= 1:
    raise ValueError("Rasio_Latih harus > 0 dan Rasio_Latih + Rasio_Validasi harus < 1.")
source = Input_CSV if Input_CSV.is_absolute() else BASE_DIR / Input_CSV
if not source.exists(): raise FileNotFoundError(f"File tidak ditemukan: {source}")
df = pd.read_csv(source, sep=Separator_CSV, dtype="string", encoding="utf-8-sig")
if Kolom_Grup:
    if Kolom_Grup not in df: raise KeyError(f"Kolom grup tidak ditemukan: {Kolom_Grup}")
    if df[Kolom_Grup].isna().any(): raise ValueError("Isi Kolom_Grup yang kosong terlebih dahulu agar semua baris bisa dibagi.")
    groups = df[Kolom_Grup].unique().tolist()
    rng = np.random.default_rng(Benih_Acak)
    rng.shuffle(groups)
    n_train = int(len(groups) * Rasio_Latih)
    n_val = int(len(groups) * Rasio_Validasi)
    train_groups, val_groups = set(groups[:n_train]), set(groups[n_train:n_train+n_val])
    train = df[df[Kolom_Grup].isin(train_groups)]
    val = df[df[Kolom_Grup].isin(val_groups)]
    test = df[~df[Kolom_Grup].isin(train_groups | val_groups)]
else:
    rng = np.random.default_rng(Benih_Acak)
    order = rng.permutation(len(df))
    n_train = int(len(df) * Rasio_Latih)
    n_val = int(len(df) * Rasio_Validasi)
    train = df.iloc[order[:n_train]]
    val = df.iloc[order[n_train:n_train+n_val]]
    test = df.iloc[order[n_train+n_val:]]

outdir = Folder_Output if Folder_Output.is_absolute() else BASE_DIR / Folder_Output
outdir.mkdir(parents=True, exist_ok=True)
for name, part in [("latih", train), ("validasi", val), ("uji", test)]:
    if name == "validasi" and Rasio_Validasi == 0: continue
    target = outdir / f"{name}.csv"
    part.to_csv(target, index=False, encoding="utf-8-sig", sep=Separator_CSV)
    print(f"{name}: {len(part):,} baris -> {target}")
print("Pembagian grup menjaga satu grup tetap di satu bagian; rasio akhir dapat sedikit berbeda karena ukuran grup.")
